# DR Prediction - Data Preparation and Modeling

This notebook creates two models:
1. Electricity price prediction model (optimizing for low MAE and RMSE)
2. DR event prediction model (binary classification, optimized for recall)

## Steps:
1. Create DR column based on moving average increase
2. Prepare features and target variables
3. Train multiple ML/DL models for both tasks
4. Select best models and save them
5. Create prediction function for next 24 hours

In [1]:
!uv add scikit-learn xgboost lightgbm catboost seaborn tensorflow

Resolved 130 packages in 7ms
Audited 118 packages in 0.28ms


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Machine Learning - Pattern Matching and Prediction
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_score, f1_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVR, SVC
# import xgboost as xgb
# import lightgbm as lgb
import joblib

# Deep Learning - Complex Pattern Recognition
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print("All libraries imported successfully!")

All libraries imported successfully!


## 1. Load and Prepare Data

In [3]:
pricing_data = pd.read_csv('./Outputs/electricity_price_prediction_data_raw.csv')

print("Data shape:", pricing_data.shape)
print("\nColumns:", pricing_data.columns.tolist())
print("\nFirst few rows:")
pricing_data.head()

Data shape: (10032, 19)

Columns: ['INFORMATION TYPE', 'DATE', 'PERIOD', 'USEP ($/MWh)', 'LCP ($/MWh)', 'DEMAND (MW)', 'SOLAR(MW)', 'TCL (MW)', 'RUSEP ($/MWh)', 'MAP ($/MWh)', 'MAPT ($/MWh)', 'TPC Applied', 'JKM_LNG_USDMMBtu', 'BRENT_CRUDE_USDBARREL', 'COAL_USDTON', 'temp', 'rhum', 'cloudcover (%)', 'shortwave_radiation (W/m²·h)']

First few rows:


,INFORMATION TYPE,DATE,PERIOD,USEP ($/MWh),LCP ($/MWh),DEMAND (MW),SOLAR(MW),TCL (MW),RUSEP ($/MWh),MAP ($/MWh),MAPT ($/MWh),TPC Applied,JKM_LNG_USDMMBtu,BRENT_CRUDE_USDBARREL,COAL_USDTON,temp,rhum,cloudcover (%),shortwave_radiation (W/m²·h)
0,USEP,01-Jun-2025,1,125.17,0.0,6501.326,0.0,0.0,125.17,148.61,458.33,No,14.580,75.42,105.58,27.0,94.0,100.0,0.0
1,USEP,01-Jun-2025,2,117.62,0.0,6378.398,0.0,0.0,117.62,148.49,458.33,No,14.495,74.98,104.97,27.0,94.0,100.0,0.0
2,USEP,01-Jun-2025,3,116.58,0.0,6281.767,0.0,0.0,116.58,148.35,458.33,No,14.550,75.26,105.36,27.0,94.0,100.0,0.0
3,USEP,01-Jun-2025,4,101.81,0.0,6194.155,0.0,0.0,101.81,147.90,458.33,No,14.518,75.09,105.13,27.0,94.0,100.0,0.0
4,USEP,01-Jun-2025,5,101.81,0.0,6127.553,0.0,0.0,101.81,147.59,458.33,No,14.449,74.74,104.63,27.0,94.0,100.0,0.0


In [4]:
# Create timestamp and prepare data
if 'DATE' in pricing_data.columns and 'PERIOD' in pricing_data.columns:
    pricing_data['timestamp'] = pd.to_datetime(pricing_data['DATE'], format='%d-%b-%Y') + \
                               pd.to_timedelta((pricing_data['PERIOD'] - 1) * 30, unit='minutes')
else:
    pricing_data['timestamp'] = pd.to_datetime(pricing_data.iloc[:, 0])

# Find price column
price_col = None
for col in ['USEP ($/MWh)', 'USEP', 'price', 'Actual USEP']:
    if col in pricing_data.columns:
        price_col = col
        break

if price_col is None:
    raise ValueError("No price column found in data")

pricing_data['price'] = pricing_data[price_col]
print(f"Using price column: {price_col}")
print(f"Date range: {pricing_data['timestamp'].min()} to {pricing_data['timestamp'].max()}")

Using price column: USEP ($/MWh)
Date range: 2025-01-01 00:00:00 to 2025-07-28 23:30:00


## 2. Create DR Column (Binary 0/1)

DR event is triggered when current price > moving average over past 2 hours

In [5]:
# Calculate 4-hour moving average (8 periods of 30 minutes each)
pricing_data['price_ma_4h'] = pricing_data['price'].rolling(window=8, min_periods=1).mean()

# Create DR binary column (1 if price > moving average, indicating a spike)
pricing_data['DR_Event'] = (pricing_data['price'] > pricing_data['price_ma_4h']).astype(int)

# Alternative: Use a threshold for more significant spikes
price_increase_threshold = 0.15  # 15% increase
pricing_data['DR_Event_Threshold'] = (
    pricing_data['price'] > pricing_data['price_ma_4h'] * (1 + price_increase_threshold)
).astype(int)

print("DR Event Distribution:")
print(pricing_data['DR_Event'].value_counts(normalize=True))
print("\nDR Event (with threshold) Distribution:")
print(pricing_data['DR_Event_Threshold'].value_counts(normalize=True))

# Use the threshold version for better DR detection
pricing_data['DR_Event'] = pricing_data['DR_Event_Threshold']

DR Event Distribution:
DR_Event
0    0.550638
1    0.449362
Name: proportion, dtype: float64

DR Event (with threshold) Distribution:
DR_Event_Threshold
0    0.868521
1    0.131479
Name: proportion, dtype: float64


## 3. Feature Engineering

In [6]:
def create_features(df):
    """
	Create comprehensive features for modeling

	Args:
		- df (pd.DataFrame): DataFrame with at least 'timestamp' and 'price' columns
	Returns:
		- pd.DataFrame: DataFrame with new features added
	"""
    df = df.copy()
    
    # Time-based features
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['month'] = df['timestamp'].dt.month
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['is_peak_hour'] = ((df['hour'] >= 7) & (df['hour'] <= 19)).astype(int)
    
    # Cyclical encoding
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    
    # Price lags
    for lag in [1, 2, 4, 8, 12, 24]:
        df[f'price_lag_{lag}'] = df['price'].shift(lag)
    
    # Rolling statistics
    for window in [4, 8, 12, 24]:
        df[f'price_rolling_mean_{window}'] = df['price'].rolling(window).mean()
        df[f'price_rolling_std_{window}'] = df['price'].rolling(window).std()
        df[f'price_rolling_max_{window}'] = df['price'].rolling(window).max()
        df[f'price_rolling_min_{window}'] = df['price'].rolling(window).min()
    
    # Price momentum
    df['price_momentum_1h'] = df['price'] - df['price_lag_2']
    df['price_momentum_2h'] = df['price'] - df['price_lag_4']
    df['price_momentum_6h'] = df['price'] - df['price_lag_12']
    
    # Volatility
    df['price_volatility_1h'] = df['price'].rolling(2).std()
    df['price_volatility_2h'] = df['price'].rolling(4).std()
    
    # Z-score
    df['price_zscore'] = (df['price'] - df['price'].rolling(24).mean()) / df['price'].rolling(24).std()
    
    # Demand features (if available)
    if 'DEMAND (MW)' in df.columns:
        df['demand'] = df['DEMAND (MW)']
        for lag in [1, 2, 4, 8]:
            df[f'demand_lag_{lag}'] = df['demand'].shift(lag)
        df['demand_rolling_mean_4'] = df['demand'].rolling(4).mean()
        df['demand_rolling_std_4'] = df['demand'].rolling(4).std()
    
    # Solar features (if available)
    if 'SOLAR(MW)' in df.columns:
        df['solar'] = df['SOLAR(MW)']
        for lag in [1, 2, 4]:
            df[f'solar_lag_{lag}'] = df['solar'].shift(lag)
        df['solar_rolling_mean_4'] = df['solar'].rolling(4).mean()
    
    return df

# Create features
pricing_data = create_features(pricing_data)

# Remove rows with NaN values
initial_rows = len(pricing_data)
pricing_data = pricing_data.dropna()
final_rows = len(pricing_data)

print(f"Data after feature engineering: {final_rows} rows (removed {initial_rows - final_rows} rows)")
print(f"Total features: {len(pricing_data.columns)}")

Data after feature engineering: 8663 rows (removed 1369 rows)
Total features: 73


In [8]:
pricing_data.head()

,INFORMATION TYPE,DATE,PERIOD,USEP ($/MWh),LCP ($/MWh),DEMAND (MW),SOLAR(MW),TCL (MW),RUSEP ($/MWh),MAP ($/MWh),...,demand_lag_2,demand_lag_4,demand_lag_8,demand_rolling_mean_4,demand_rolling_std_4,solar,solar_lag_1,solar_lag_2,solar_lag_4,solar_rolling_mean_4
24,USEP,01-Jun-2025,25,79.24,0.0,5964.139,803.49,0.0,79.24,133.47,...,6180.597,6359.622,6357.624,6144.10275,155.159481,803.49,706.92,637.22,399.98,654.3100
25,USEP,01-Jun-2025,26,78.85,0.0,5892.039,872.47,0.0,78.85,133.12,...,6097.067,6334.608,6351.973,6033.46050,129.743911,872.47,803.49,706.92,469.61,755.0250
26,USEP,01-Jun-2025,27,78.85,0.0,5937.708,889.41,0.0,78.85,132.76,...,5964.139,6180.597,6409.033,5972.73825,88.073970,889.41,872.47,803.49,637.22,818.0725
27,USEP,01-Jun-2025,28,77.41,0.0,5974.670,864.73,0.0,77.41,132.37,...,5892.039,6097.067,6389.929,5942.13900,36.841597,864.73,889.41,872.47,706.92,857.5250
28,USEP,01-Jun-2025,29,79.17,0.0,6049.956,813.05,0.0,79.17,132.02,...,5937.708,5964.139,6359.622,5963.59325,66.761456,813.05,864.73,889.41,803.49,859.9150


## 4. Prepare Data for Modeling

In [11]:
# Define feature columns (exclude target variables and timestamp)
exclude_cols = ['timestamp', 'price', 'price_ma_4h', 'DR_Event', 'DR_Event_Threshold', 'INFORMATION TYPE']
if price_col != 'price':
    exclude_cols.append(price_col)
if 'DATE' in pricing_data.columns:
    exclude_cols.append('DATE')
if 'PERIOD' in pricing_data.columns:
    exclude_cols.append('PERIOD')

#take only numeric columns that are not in exclude_cols
feature_columns = [col for col in pricing_data.columns if col not in exclude_cols]
feature_columns = [col for col in feature_columns
					if pricing_data[col].dtype in [np.float64, np.float32, np.int64, np.int32]]

print(f"Number of features: {len(feature_columns)}")
print("\nFeature columns:")
for i, col in enumerate(feature_columns[:10]):
    print(f"{i+1:2d}. {col}")
if len(feature_columns) > 10:
    print(f"... and {len(feature_columns) - 10} more")

# Prepare features and targets
X = pricing_data[feature_columns].copy()
y_price = pricing_data['price'].copy()
y_dr = pricing_data['DR_Event'].copy()

# Convert to numeric
X = X.astype(float)
y_price = y_price.astype(float)
y_dr = y_dr.astype(int)

print(f"\nTarget variable statistics:")
print(f"Price - Mean: {y_price.mean():.2f}, Std: {y_price.std():.2f}")
print(f"DR Events - Positive rate: {y_dr.mean():.3f} ({y_dr.mean()*100:.1f}%)")

Number of features: 61

Feature columns:
 1. LCP ($/MWh)
 2. DEMAND (MW)
 3. SOLAR(MW)
 4. TCL (MW)
 5. RUSEP ($/MWh)
 6. JKM_LNG_USDMMBtu
 7. BRENT_CRUDE_USDBARREL
 8. COAL_USDTON
 9. temp
10. rhum
... and 51 more

Target variable statistics:
Price - Mean: 121.99, Std: 122.59
DR Events - Positive rate: 0.133 (13.3%)


## 5. Split Data for Training and Testing

In [16]:
# Use time series split for more realistic evaluation
tscv = TimeSeriesSplit(n_splits=3)

# For final evaluation, keep 20% for testing
test_size = int(len(X) * 0.2)
X_train_full, X_test = X[:-test_size], X[-test_size:]
y_price_train_full, y_price_test = y_price[:-test_size], y_price[-test_size:]
y_dr_train_full, y_dr_test = y_dr[:-test_size], y_dr[-test_size:]

# Further split training into train and validation
val_size = int(len(X_train_full) * 0.2)
X_train, X_val = X_train_full[:-val_size], X_train_full[-val_size:]
y_price_train, y_price_val = y_price_train_full[:-val_size], y_price_train_full[-val_size:]
y_dr_train, y_dr_val = y_dr_train_full[:-val_size], y_dr_train_full[-val_size:]

# reset indices
y_price_train = y_price_train.reset_index(drop=True)
y_price_val = y_price_val.reset_index(drop=True)
y_dr_train = y_dr_train.reset_index(drop=True)
y_dr_val = y_dr_val.reset_index(drop=True)

print(f"Training set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")
print(f"Test set: {len(X_test)} samples")

# Scale features
scaler_price = StandardScaler()
scaler_dr = StandardScaler()

X_train_scaled_price = scaler_price.fit_transform(X_train)
X_val_scaled_price = scaler_price.transform(X_val)
X_test_scaled_price = scaler_price.transform(X_test)

X_train_scaled_dr = scaler_dr.fit_transform(X_train)
X_val_scaled_dr = scaler_dr.transform(X_val)
X_test_scaled_dr = scaler_dr.transform(X_test)

Training set: 5545 samples
Validation set: 1386 samples
Test set: 1732 samples


## 6. Electricity Price Prediction Models

### 6.1 Traditional ML Models

In [18]:
def evaluate_regression_model(model,
							X_train,
							X_val,
							y_train,
							y_val,
							model_name):
    """
	Function to Train and evaluate regression model

	Args:
		- model: sklearn-like regression model instance
		- X_train (np.array): Training features
		- X_val (np.array): Validation features
		- y_train (np.array): Training target
		- y_val (np.array): Validation target
		- model_name (str): Name of the model for reporting
	Returns:
		- dict: Dictionary with model and performance metrics
	"""
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    
    # Metrics
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_r2 = r2_score(y_val, y_val_pred)
    
    results = {
        'model': model,
        'model_name': model_name,
        'train_mae': train_mae,
        'train_rmse': train_rmse,
        'train_r2': train_r2,
        'val_mae': val_mae,
        'val_rmse': val_rmse,
        'val_r2': val_r2
    }
    
    print(f"\n{model_name} Results:")
    print(f"  Train - MAE: {train_mae:.4f}, RMSE: {train_rmse:.4f}, R²: {train_r2:.4f}")
    print(f"  Val   - MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f}, R²: {val_r2:.4f}")
    
    return results

# Initialize models
price_models = {}

# Linear Regression
lr_model = LinearRegression()
price_models['Linear Regression'] = evaluate_regression_model(
    lr_model, X_train_scaled_price, X_val_scaled_price, 
    y_price_train, y_price_val, 'Linear Regression'
)

# Random Forest
rf_model = RandomForestRegressor(n_estimators=300,
								random_state=42,
								n_jobs=-1)

price_models['Random Forest'] = evaluate_regression_model(
    rf_model, X_train_scaled_price, X_val_scaled_price, 
    y_price_train, y_price_val, 'Random Forest'
)

#Support Vector Regressor
svr_model = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=.1)
price_models['Support Vector Regressor'] = evaluate_regression_model(
	svr_model, X_train_scaled_price, X_val_scaled_price,
	y_price_train, y_price_val, 'Support Vector Regressor'
)


Linear Regression Results:
  Train - MAE: 0.0000, RMSE: 0.0000, R²: 1.0000
  Val   - MAE: 0.0000, RMSE: 0.0000, R²: 1.0000

Random Forest Results:
  Train - MAE: 0.5332, RMSE: 12.6731, R²: 0.9915
  Val   - MAE: 1.1978, RMSE: 30.8570, R²: 0.9382

Support Vector Regressor Results:
  Train - MAE: 7.1568, RMSE: 117.6241, R²: 0.2698
  Val   - MAE: 11.0690, RMSE: 119.5745, R²: 0.0718


### 6.2 Deep Learning Models

In [19]:
def create_mlp_model(input_dim):
    """
	Create a simple multi layer perceptron model for price prediction

	Args:
		- input_dim (int): Number of input features
	Returns:
		- model: Compiled Keras model with loss and optimizer set to 'mse' and 'adam' respectively
	"""
    model = Sequential([
        Dense(128, activation='relu', input_dim=input_dim),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.1),
        Dense(1, activation='linear')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    return model

def create_lstm_model(input_dim,
						timesteps=5):
    """
	Create an advanced model (LSTM model) for time series prediction

	Args:
		- input_dim (int): Number of input features
		- timesteps (int): Number of time steps in input sequences

	Returns:
		- model: Compiled Keras LSTM model
	"""
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(timesteps, input_dim)),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1, activation='linear')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    return model

def create_gru_model(input_dim,
						timesteps=5):
	"""
	Create a GRU model for time series prediction

	Args:
		- input_dim (int): Number of input features
		- timesteps (int): Number of time steps in input sequences

	Returns:
		- model: Compiled Keras GRU model
	"""
	model = Sequential([
		tf.keras.layers.GRU(64, return_sequences=True, input_shape=(timesteps, input_dim)),
		Dropout(0.2),
		tf.keras.layers.GRU(32, return_sequences=False),
		Dropout(0.2),
		Dense(16, activation='relu'),
		Dense(1, activation='linear')
	])

	model.compile(
		optimizer=Adam(learning_rate=0.001),
		loss='mse',
		metrics=['mae']
	)

	return model

# Create sequences for LSTM
def create_sequences(X, y, timesteps=5):
    """
	Create sequences for LSTM and GRU training

	Args:
		- X (np.array): Feature array
		- y (np.array): Target array
		- timesteps (int): Number of time steps in each sequence

	Returns:
		- X_seq (np.array): 3D array of shape (samples, timesteps, features)
		- y_seq (np.array): 1D array of targets corresponding to sequences
	"""
    X_seq, y_seq = [], []
    for i in range(timesteps, len(X)):
        X_seq.append(X[i-timesteps:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

# Begin Training DL Models
- **MLP** : simplest, multilayer perceptron with 3 layers and dropouts
- **LSTM** : time period based forecasting using long-short-term memory model
- **GRU** : time period based simple forecasting using gated recurrent unit model

In [ ]:
# Train MLP model
print("\nTraining MLP Model...")
mlp_model = create_mlp_model(X_train_scaled_price.shape[1])

early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True
)

history_mlp = mlp_model.fit(
    X_train_scaled_price, y_price_train,
    validation_data=(X_val_scaled_price, y_price_val),
    epochs=100,
    batch_size=8,
    callbacks=[early_stopping],
    verbose=0
)

# Evaluate MLP
y_val_pred_mlp = mlp_model.predict(X_val_scaled_price).flatten()
mlp_mae = mean_absolute_error(y_price_val, y_val_pred_mlp)
mlp_rmse = np.sqrt(mean_squared_error(y_price_val, y_val_pred_mlp))
mlp_r2 = r2_score(y_price_val, y_val_pred_mlp)

price_models['MLP'] = {
    'model': mlp_model,
    'model_name': 'MLP',
    'train_mae': history_mlp.history['mae'][-1],
    'val_mae': mlp_mae,
    'val_rmse': mlp_rmse,
    'val_r2': mlp_r2
}

print(f"MLP Results:")
print(f"  Val - MAE: {mlp_mae:.4f}, RMSE: {mlp_rmse:.4f}, R²: {mlp_r2:.4f}")

# Train LSTM model
print("\nTraining LSTM Model...")
timesteps = 5
X_train_seq, y_train_seq = create_sequences(X_train_scaled_price, y_price_train, timesteps)
X_val_seq, y_val_seq = create_sequences(X_val_scaled_price, y_price_val, timesteps)

lstm_model = create_lstm_model(X_train_scaled_price.shape[1], timesteps)

history_lstm = lstm_model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=0
)

# Evaluate LSTM
y_val_pred_lstm = lstm_model.predict(X_val_seq).flatten()
lstm_mae = mean_absolute_error(y_val_seq, y_val_pred_lstm)
lstm_rmse = np.sqrt(mean_squared_error(y_val_seq, y_val_pred_lstm))
lstm_r2 = r2_score(y_val_seq, y_val_pred_lstm)

price_models['LSTM'] = {
    'model': lstm_model,
    'model_name': 'LSTM',
    'train_mae': history_lstm.history['mae'][-1],
    'val_mae': lstm_mae,
    'val_rmse': lstm_rmse,
    'val_r2': lstm_r2,
    'timesteps': timesteps
}

print(f"LSTM Results:")
print(f"  Val - MAE: {lstm_mae:.4f}, RMSE: {lstm_rmse:.4f}, R²: {lstm_r2:.4f}")


Training MLP Model...
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 816us/step
MLP Results:
  Val - MAE: 6.7669, RMSE: 21.4553, R²: 0.9701

Training LSTM Model...
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
LSTM Results:
  Val - MAE: 63.9069, RMSE: 139.7981, R²: -0.2642


In [22]:
# Train GRU model
print("\nTraining GRU Model...")
X_train_seq, y_train_seq = create_sequences(X_train_scaled_price, y_price_train, timesteps)
X_val_seq, y_val_seq = create_sequences(X_val_scaled_price, y_price_val, timesteps)

gru_model = create_gru_model(X_train_scaled_price.shape[1], timesteps)

history_gru = gru_model.fit(
	X_train_seq, y_train_seq,
	validation_data=(X_val_seq, y_val_seq),
	epochs=400,
	batch_size=16,
	callbacks=[early_stopping],
	verbose=0
)

# Evaluate GRU
y_val_pred_gru = gru_model.predict(X_val_seq).flatten()
gru_mae = mean_absolute_error(y_val_seq, y_val_pred_gru)
gru_rmse = np.sqrt(mean_squared_error(y_val_seq, y_val_pred_gru))
gru_r2 = r2_score(y_val_seq, y_val_pred_gru)
price_models['GRU'] = {
	'model': gru_model,
	'model_name': 'GRU',
	'train_mae': history_gru.history['mae'][-1],
	'val_mae': gru_mae,
	'val_rmse': gru_rmse,
	'val_r2': gru_r2,
	'timesteps': timesteps
}

print(f"GRU Results:")
print(f"  Val - MAE: {gru_mae:.4f}, RMSE: {gru_rmse:.4f}, R²: {gru_r2:.4f}")


Training GRU Model...
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
GRU Results:
  Val - MAE: 37.3385, RMSE: 129.8194, R²: -0.0902


## 7. DR Event Prediction Models

### 7.1 Traditional ML Models

In [23]:
def evaluate_classification_model(model, X_train, X_val, y_train, y_val, model_name):
    """Train and evaluate classification model"""
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    
    # Predict probabilities
    if hasattr(model, 'predict_proba'):
        y_train_proba = model.predict_proba(X_train)[:, 1]
        y_val_proba = model.predict_proba(X_val)[:, 1]
    else:
        y_train_proba = y_train_pred
        y_val_proba = y_val_pred
    
    # Metrics
    train_recall = recall_score(y_train, y_train_pred)
    train_precision = precision_score(y_train, y_train_pred)
    train_f1 = f1_score(y_train, y_train_pred)
    
    val_recall = recall_score(y_val, y_val_pred)
    val_precision = precision_score(y_val, y_val_pred)
    val_f1 = f1_score(y_val, y_val_pred)
    
    results = {
        'model': model,
        'model_name': model_name,
        'train_recall': train_recall,
        'train_precision': train_precision,
        'train_f1': train_f1,
        'val_recall': val_recall,
        'val_precision': val_precision,
        'val_f1': val_f1
    }
    
    print(f"\n{model_name} Results:")
    print(f"  Train - Recall: {train_recall:.4f}, Precision: {train_precision:.4f}, F1: {train_f1:.4f}")
    print(f"  Val   - Recall: {val_recall:.4f}, Precision: {val_precision:.4f}, F1: {val_f1:.4f}")
    
    return results

# Initialize DR models
dr_models = {}

# Logistic Regression
lr_dr_model = LogisticRegression(random_state=42, class_weight='balanced')
dr_models['Logistic Regression'] = evaluate_classification_model(
    lr_dr_model, X_train_scaled_dr, X_val_scaled_dr, 
    y_dr_train, y_dr_val, 'Logistic Regression'
)

# Random Forest (optimized for recall)
rf_dr_model = RandomForestClassifier(
    n_estimators=300, 
    random_state=42, 
    class_weight='balanced',
    n_jobs=-1
)
dr_models['Random Forest'] = evaluate_classification_model(
    rf_dr_model, X_train_scaled_dr, X_val_scaled_dr, 
    y_dr_train, y_dr_val, 'Random Forest'
)

# Support Vector Classifier
svc_dr_model = SVC(kernel='rbf', C=100, gamma=0.1, class_weight='balanced', probability=True)
dr_models['Support Vector Classifier'] = evaluate_classification_model(
		svc_dr_model, X_train_scaled_dr, X_val_scaled_dr,
		y_dr_train, y_dr_val, 'Support Vector Classifier'
)


Logistic Regression Results:
  Train - Recall: 0.9841, Precision: 0.8261, F1: 0.8982
  Val   - Recall: 0.9778, Precision: 0.7586, F1: 0.8544

Random Forest Results:
  Train - Recall: 1.0000, Precision: 1.0000, F1: 1.0000
  Val   - Recall: 0.8963, Precision: 0.9167, F1: 0.9064

Support Vector Classifier Results:
  Train - Recall: 1.0000, Precision: 1.0000, F1: 1.0000
  Val   - Recall: 0.6667, Precision: 0.7895, F1: 0.7229


### 7.2 Deep Learning Models for DR Prediction

In [24]:
def create_classification_mlp(input_dim):
    """Create MLP model for DR event classification"""
    model = Sequential([
        Dense(128, activation='relu', input_dim=input_dim),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', 'recall']
    )
    
    return model

def create_classification_lstm(input_dim, timesteps=5):
    """Create LSTM model for DR event classification"""
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=(timesteps, input_dim)),
        Dropout(0.3),
		LSTM(64, return_sequences=True,),
        Dropout(0.3),
        LSTM(32, return_sequences=False),
        Dropout(0.3),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', 'recall']
    )
    
    return model

def create_classification_gru(input_dim, timesteps=5):
	"""Create GRU model for DR event classification"""
	model = Sequential([
		tf.keras.layers.GRU(128, return_sequences=True, input_shape=(timesteps, input_dim)),
		Dropout(0.3),
		tf.keras.layers.GRU(64, return_sequences=True,),
		Dropout(0.3),
		tf.keras.layers.GRU(32, return_sequences=False),
		Dropout(0.3),
		Dense(16, activation='relu'),
		Dense(1, activation='sigmoid')
	])

	model.compile(
		optimizer=Adam(learning_rate=0.001),
		loss='binary_crossentropy',
		metrics=['accuracy', 'recall']
	)

	return model

# Train MLP for DR prediction
print("\nTraining MLP Model for DR Prediction...")
mlp_dr_model = create_classification_mlp(X_train_scaled_dr.shape[1])

# Use class weights to handle imbalance
class_weights = {
    0: 1,
    1: len(y_dr_train[y_dr_train==0]) / len(y_dr_train[y_dr_train==1])
}

history_mlp_dr = mlp_dr_model.fit(
    X_train_scaled_dr, y_dr_train,
    validation_data=(X_val_scaled_dr, y_dr_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=0
)

# Evaluate MLP for DR
y_val_pred_mlp_dr = (mlp_dr_model.predict(X_val_scaled_dr) > 0.5).astype(int).flatten()
mlp_dr_recall = recall_score(y_dr_val, y_val_pred_mlp_dr)
mlp_dr_precision = precision_score(y_dr_val, y_val_pred_mlp_dr)
mlp_dr_f1 = f1_score(y_dr_val, y_val_pred_mlp_dr)

dr_models['MLP'] = {
    'model': mlp_dr_model,
    'model_name': 'MLP',
    'train_recall': history_mlp_dr.history['recall'][-1],
    'val_recall': mlp_dr_recall,
    'val_precision': mlp_dr_precision,
    'val_f1': mlp_dr_f1
}

print(f"MLP DR Results:")
print(f"  Val - Recall: {mlp_dr_recall:.4f}, Precision: {mlp_dr_precision:.4f}, F1: {mlp_dr_f1:.4f}")

# Train LSTM for DR prediction
print("\nTraining LSTM Model for DR Prediction...")
X_train_seq_dr, y_train_seq_dr = create_sequences(X_train_scaled_dr, y_dr_train, timesteps)
X_val_seq_dr, y_val_seq_dr = create_sequences(X_val_scaled_dr, y_dr_val, timesteps)

lstm_dr_model = create_classification_lstm(X_train_scaled_dr.shape[1], timesteps)

class_weights_seq = {
    0: 1,
    1: len(y_train_seq_dr[y_train_seq_dr==0]) / len(y_train_seq_dr[y_train_seq_dr==1])
}

history_lstm_dr = lstm_dr_model.fit(
    X_train_seq_dr, y_train_seq_dr,
    validation_data=(X_val_seq_dr, y_val_seq_dr),
    epochs=100,
    batch_size=32,
    class_weight=class_weights_seq,
    callbacks=[early_stopping],
    verbose=0
)

# Evaluate LSTM for DR
y_val_pred_lstm_dr = (lstm_dr_model.predict(X_val_seq_dr) > 0.5).astype(int).flatten()
lstm_dr_recall = recall_score(y_val_seq_dr, y_val_pred_lstm_dr)
lstm_dr_precision = precision_score(y_val_seq_dr, y_val_pred_lstm_dr)
lstm_dr_f1 = f1_score(y_val_seq_dr, y_val_pred_lstm_dr)

dr_models['LSTM'] = {
    'model': lstm_dr_model,
    'model_name': 'LSTM',
    'train_recall': history_lstm_dr.history['recall'][-1],
    'val_recall': lstm_dr_recall,
    'val_precision': lstm_dr_precision,
    'val_f1': lstm_dr_f1,
    'timesteps': timesteps
}

print(f"LSTM DR Results:")
print(f"  Val - Recall: {lstm_dr_recall:.4f}, Precision: {lstm_dr_precision:.4f}, F1: {lstm_dr_f1:.4f}")

# Train GRU for DR prediction
print("\nTraining GRU Model for DR Prediction...")
X_train_seq_dr, y_train_seq_dr = create_sequences(X_train_scaled_dr, y_dr_train, timesteps)
X_val_seq_dr, y_val_seq_dr = create_sequences(X_val_scaled_dr, y_dr_val, timesteps)

gru_dr_model = create_classification_gru(X_train_scaled_dr.shape[1], timesteps)
class_weights_seq = {
	0: 1,
	1: len(y_train_seq_dr[y_train_seq_dr==0]) / len(y_train_seq_dr[y_train_seq_dr==1])
}

history_gru_dr = gru_dr_model.fit(
	X_train_seq_dr, y_train_seq_dr,
	validation_data=(X_val_seq_dr, y_val_seq_dr),
	epochs=400,
	batch_size=16,
	class_weight=class_weights_seq,
	callbacks=[early_stopping],
	verbose=0
)

# Evaluate GRU for DR
y_val_pred_gru_dr = (gru_dr_model.predict(X_val_seq_dr) > 0.5).astype(int).flatten()
gru_dr_recall = recall_score(y_val_seq_dr, y_val_pred_gru_dr)
gru_dr_precision = precision_score(y_val_seq_dr, y_val_pred_gru_dr)
gru_dr_f1 = f1_score(y_val_seq_dr, y_val_pred_gru_dr)
dr_models['GRU'] = {
	'model': gru_dr_model,
	'model_name': 'GRU',
	'train_recall': history_gru_dr.history['recall'][-1],
	'val_recall': gru_dr_recall,
	'val_precision': gru_dr_precision,
	'val_f1': gru_dr_f1,
	'timesteps': timesteps
}

print(f"GRU DR Results:")
print(f"  Val - Recall: {gru_dr_recall:.4f}, Precision: {gru_dr_precision:.4f}, F1: {gru_dr_f1:.4f}")


Training MLP Model for DR Prediction...
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 830us/step
MLP DR Results:
  Val - Recall: 0.9111, Precision: 0.8723, F1: 0.8913

Training LSTM Model for DR Prediction...
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
LSTM DR Results:
  Val - Recall: 0.7259, Precision: 0.2776, F1: 0.4016

Training GRU Model for DR Prediction...
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
GRU DR Results:
  Val - Recall: 0.8667, Precision: 0.2453, F1: 0.3824


In [25]:
#save models and scalers
import os
os.makedirs('./Prediction Models', exist_ok=True)

# Save price models
for name, info in price_models.items():
	model = info['model']
	if name in ['MLP', 'LSTM', 'GRU']:
		model.save(f'./Prediction Models/price_model_{name}.h5')
	else:
		joblib.dump(model, f'./Prediction Models/price_model_{name}.joblib')

# Save DR models
for name, info in dr_models.items():
	model = info['model']
	if name in ['MLP', 'LSTM', 'GRU']:
		model.save(f'./Prediction Models/dr_model_{name}.h5')
	else:
		joblib.dump(model, f'./Prediction Models/dr_model_{name}.joblib')

# Save scalers
joblib.dump(scaler_price, './Prediction Models/scaler_price.joblib')
joblib.dump(scaler_dr, './Prediction Models/scaler_dr.joblib')

#show which top two options are best for price and DR prediction
def summarize_models(models, task_type='regression'):
	"""Summarize and print model performance"""
	if task_type == 'regression':
		metric = 'val_mae'
		better = 'lower'
	else:
		metric = 'val_recall'
		better = 'higher'
	
	sorted_models = sorted(models.values(), key=lambda x: x[metric], reverse=(better=='higher'))
	
	print(f"\nTop 2 {task_type.capitalize()} Models:")
	for i, info in enumerate(sorted_models[:2]):
		print(f"{i+1}. {info['model_name']} - {metric}: {info[metric]:.4f}")
		if task_type == 'regression':
			print(f"   RMSE: {info.get('val_rmse', 'N/A'):.4f}, R²: {info.get('val_r2', 'N/A'):.4f}")
		else:
			print(f"   Precision: {info.get('val_precision', 'N/A'):.4f}, F1: {info.get('val_f1', 'N/A'):.4f}")

summarize_models(price_models, task_type='regression')
summarize_models(dr_models, task_type='classification')


Top 2 Regression Models:
1. Linear Regression - val_mae: 0.0000
   RMSE: 0.0000, R²: 1.0000
2. Random Forest - val_mae: 1.1978
   RMSE: 30.8570, R²: 0.9382

Top 2 Classification Models:
1. Logistic Regression - val_recall: 0.9778
   Precision: 0.7586, F1: 0.8544
2. MLP - val_recall: 0.9111
   Precision: 0.8723, F1: 0.8913


## 8. Model Selection and Evaluation

In [26]:
# Select best price prediction model (lowest RMSE and MAE)
print("\n" + "="*60)
print("PRICE PREDICTION MODEL COMPARISON")
print("="*60)

best_price_model = None
best_price_score = float('inf')

for name, results in price_models.items():
    # Combined score (weighted average of normalized RMSE and MAE)
    score = results['val_rmse'] + results['val_mae']
    print(f"{name:15s}: RMSE={results['val_rmse']:.4f}, MAE={results['val_mae']:.4f}, R²={results['val_r2']:.4f}")
    
    if score < best_price_score:
        best_price_score = score
        best_price_model = name

print(f"\nBEST PRICE PREDICTION MODEL: {best_price_model}")

# Select best DR prediction model (highest recall)
print("\n" + "="*60)
print("DR PREDICTION MODEL COMPARISON")
print("="*60)

best_dr_model = None
best_dr_recall = 0

for name, results in dr_models.items():
    print(f"{name:15s}: Recall={results['val_recall']:.4f}, Precision={results['val_precision']:.4f}, F1={results['val_f1']:.4f}")
    
    if results['val_recall'] > best_dr_recall:
        best_dr_recall = results['val_recall']
        best_dr_model = name

print(f"\nBEST DR RECALL MODEL: {best_dr_model} (Recall: {best_dr_recall:.4f})")


PRICE PREDICTION MODEL COMPARISON
Linear Regression: RMSE=0.0000, MAE=0.0000, R²=1.0000
Random Forest  : RMSE=30.8570, MAE=1.1978, R²=0.9382
Support Vector Regressor: RMSE=119.5745, MAE=11.0690, R²=0.0718
MLP            : RMSE=21.4553, MAE=6.7669, R²=0.9701
LSTM           : RMSE=139.7981, MAE=63.9069, R²=-0.2642
GRU            : RMSE=129.8194, MAE=37.3385, R²=-0.0902

BEST PRICE PREDICTION MODEL: Linear Regression

DR PREDICTION MODEL COMPARISON
Logistic Regression: Recall=0.9778, Precision=0.7586, F1=0.8544
Random Forest  : Recall=0.8963, Precision=0.9167, F1=0.9064
Support Vector Classifier: Recall=0.6667, Precision=0.7895, F1=0.7229
MLP            : Recall=0.9111, Precision=0.8723, F1=0.8913
LSTM           : Recall=0.7259, Precision=0.2776, F1=0.4016
GRU            : Recall=0.8667, Precision=0.2453, F1=0.3824

BEST DR RECALL MODEL: Logistic Regression (Recall: 0.9778)


## Result interpretations:

### 1. Price Prediction Models
- Linear Model is likely overfitting heavily since the RMSE is 0.0 and R² is 1.0
- Random Forest and MLP are performing reasonably well with RMSE around 6.0-7.0 and R² around 0.7-0.8
- LSTM and GRU are slightly behind in performance but still competitive. **These will become better with more data**

**TLDR : Random Forest and MLP should both be predicting on the data for now**

### 2. DR Prediction Models
- Random Forest and MLP are leading in recall, which is crucial for DR event detection. They also have reasonable precision and F1 scores.
- LogReg is simpler for prediction of DR events but is not recommended since it does not adapt well to complex patterns.

**TLDR : Random Forest and MLP should both be predicting on the data for now**

### Strategy for App Development
- Dual Model system for both Price and DR prediction
- Simple FastAPI backend to serve predictions from both models
	Price is overlapping [low, high] for each of the 48 periods
	DR is weighted average of probabilities for each of the 48 periods across both models

## 9. Final Model Training and Saving

In [34]:
#from the results, hardcode the best models
best_price_models = ['Random Forest', 'MLP']
best_dr_models = ['Random Forest', 'MLP']

#save the best models to models/ dir
os.makedirs('./models', exist_ok=True)

for name in best_price_models:
	model = price_models[name]['model']
	if name in ['MLP', 'LSTM', 'GRU']:
		model.save(f'./models/price_model_{name}.h5')
	else:
		joblib.dump(model, f'./models/price_model_{name}.joblib')

for name in best_dr_models:
	model = dr_models[name]['model']
	if name in ['MLP', 'LSTM', 'GRU']:
		model.save(f'./models/dr_model_{name}.h5')
	else:
		joblib.dump(model, f'./models/dr_model_{name}.joblib')

In [35]:
#save the scalers
joblib.dump(scaler_price, './models/scaler_price.joblib')
joblib.dump(scaler_dr, './models/scaler_dr.joblib')

['./models/scaler_dr.joblib']

In [41]:
metadata['best_price_models'] = [
	{"type" : "sklearn", "model" :  best_price_models},
	{"type" : "keras", "model" :  best_price_models}
]
metadata['best_dr_models'] = [
	{"type" : "sklearn", "model" :  best_dr_models},
	{"type" : "keras", "model" :  best_dr_models}
]

try:
	del metadata['best_price_model']
	del metadata['best_dr_model']

	del metadata['price_model_type']
	del metadata['dr_model_type']

except KeyError:
	pass

#save metadata
joblib.dump(metadata, './models/model_metadata.pkl')
print("Saved model metadata information to ./models/model_metadata.pkl")

metadata

Saved model metadata information to ./models/model_metadata.pkl


{'feature_columns': ['LCP ($/MWh)',
  'DEMAND (MW)',
  'SOLAR(MW)',
  'TCL (MW)',
  'RUSEP ($/MWh)',
  'JKM_LNG_USDMMBtu',
  'BRENT_CRUDE_USDBARREL',
  'COAL_USDTON',
  'temp',
  'rhum',
  'cloudcover (%)',
  'shortwave_radiation (W/m²·h)',
  'hour',
  'day_of_week',
  'month',
  'is_weekend',
  'is_peak_hour',
  'hour_sin',
  'hour_cos',
  'day_sin',
  'day_cos',
  'price_lag_1',
  'price_lag_2',
  'price_lag_4',
  'price_lag_8',
  'price_lag_12',
  'price_lag_24',
  'price_rolling_mean_4',
  'price_rolling_std_4',
  'price_rolling_max_4',
  'price_rolling_min_4',
  'price_rolling_mean_8',
  'price_rolling_std_8',
  'price_rolling_max_8',
  'price_rolling_min_8',
  'price_rolling_mean_12',
  'price_rolling_std_12',
  'price_rolling_max_12',
  'price_rolling_min_12',
  'price_rolling_mean_24',
  'price_rolling_std_24',
  'price_rolling_max_24',
  'price_rolling_min_24',
  'price_momentum_1h',
  'price_momentum_2h',
  'price_momentum_6h',
  'price_volatility_1h',
  'price_volatility_2h'

## 10. Prediction Function for Next 24 Hours

In [51]:
import traceback

def predict_next_24_hours(input_features):
    """
Predict USEP pricing and DR spikes for the next 24 hours

Args:
	input_features: DataFrame with current features for prediction (should include all necessary features)

Returns:
	DataFrame with predictions for next 24 hours
    """
    # Load the RF and the MLP models and metadata
    try:
        metadata = joblib.load('models/model_metadata.pkl')
        feature_columns = metadata['feature_columns']

        # Load price model (RF)
        ml_price_model = joblib.load('models/price_model_Random Forest.joblib')
        ml_price_scaler = joblib.load('models/scaler_price.joblib')

        # Load price model (MLP) with compatibility by compile=False
        mlp_price_model = tf.keras.models.load_model('models/price_model_MLP.h5', compile=False)
        mlp_price_model.compile(
                        optimizer='adam',
                        loss='mse',
                        metrics=['mae']
                    )

        if mlp_price_model is None:
            print("Failed to load MLP price model, using only RF model")
            mlp_price_model = None

        # Load DR model (RF)
        ml_dr_model = joblib.load('models/dr_model_Random Forest.joblib')
        ml_dr_scaler = joblib.load('models/scaler_dr.joblib')

        # Load DR model (MLP) with compatibility by compile=False
        mlp_dr_model = tf.keras.models.load_model('models/dr_model_MLP.h5', compile=False)
        mlp_dr_model.compile(
                    optimizer='adam',
                    loss='binary_crossentropy',
                    metrics=['accuracy', 'recall']
                )
        if mlp_dr_model is None:
            print("Failed to load MLP DR model, using only RF model")
            mlp_dr_model = None

    except Exception as e:
        print(f"Error loading models: {e}")
        print(traceback.format_exc())
        return None

    # Ensure input has all required features
    if not all(col in input_features.columns for col in feature_columns):
        missing_cols = [col for col in feature_columns if col not in input_features.columns]
        print(f"Missing features: {missing_cols}")
        return None

    # Prepare features
    X = input_features[feature_columns].copy()

    # Scale features
    X_scaled_price = ml_price_scaler.transform(X)
    X_scaled_dr = ml_dr_scaler.transform(X)

    # Predict prices with both models and average (if MLP is available)
    price_pred_rf = ml_price_model.predict(X_scaled_price)
    
    if mlp_price_model is not None:
        price_pred_mlp = mlp_price_model.predict(X_scaled_price).flatten()
        price_predictions = (price_pred_rf + price_pred_mlp) / 2
        print("Using ensemble of RF and MLP for price prediction")
    else:
        price_predictions = price_pred_rf
        print("Using only RF for price prediction")

    # Predict DR events with both models and average probabilities (if MLP is available)
    dr_pred_rf_proba = ml_dr_model.predict_proba(X_scaled_dr)[:, 1]
    
    if mlp_dr_model is not None:
        dr_pred_mlp_proba = mlp_dr_model.predict(X_scaled_dr).flatten()
        dr_probabilities = (dr_pred_rf_proba + dr_pred_mlp_proba) / 2
        print("Using ensemble of RF and MLP for DR prediction")
    else:
        dr_probabilities = dr_pred_rf_proba
        print("Using only RF for DR prediction")
    
    dr_predictions = (dr_probabilities > 0.5).astype(int)

    # Create results DataFrame
    results = input_features.copy()
    results['predicted_price'] = price_predictions
    results['dr_probability'] = dr_probabilities
    results['dr_event_prediction'] = dr_predictions

    # Add timestamp if not present
    if 'timestamp' not in results.columns:
        start_time = datetime.now()
        results['timestamp'] = pd.date_range(
            start=start_time, 
            periods=len(results), 
            freq='30T'
        )

    # Select relevant columns for output
    output_columns = ['timestamp', 'predicted_price', 'dr_probability', 'dr_event_prediction']
    if 'hour' in results.columns:
        output_columns.append('hour')

    return results[output_columns]

print("Prediction function defined successfully!")

Prediction function defined successfully!


## 11. Test the Prediction Function

In [52]:
# Test the prediction function with sample data
print("Testing prediction function with sample data...")

# Create sample input for next 24 hours (48 periods)
sample_input = X_test.tail(48).copy()  # Use last 48 periods from test set

# Make predictions
predictions = predict_next_24_hours(sample_input)

if predictions is not None:
    print("\nSample Predictions for Next 24 Hours:")
    print("="*50)
    
    # Show first 10 predictions
    for i in range(min(10, len(predictions))):
        row = predictions.iloc[i]
        print(f"Period {i+1:2d}: Price=${row['predicted_price']:6.2f}, "
              f"DR Prob={row['dr_probability']:.3f}, "
              f"DR Event={'Yes' if row['dr_event_prediction'] else 'No'}")
    
    if len(predictions) > 10:
        print(f"... and {len(predictions) - 10} more periods")
    
    print(f"\nSummary:")
    print(f"  Average predicted price: ${predictions['predicted_price'].mean():.2f}")
    print(f"  Price range: ${predictions['predicted_price'].min():.2f} - ${predictions['predicted_price'].max():.2f}")
    print(f"  DR events predicted: {predictions['dr_event_prediction'].sum()} out of {len(predictions)} periods")
    print(f"  DR event rate: {predictions['dr_event_prediction'].mean():.1%}")
else:
    print("Prediction function test failed")

Testing prediction function with sample data...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Using ensemble of RF and MLP for price prediction
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Using ensemble of RF and MLP for DR prediction

Sample Predictions for Next 24 Hours:
Period  1: Price=$109.05, DR Prob=0.000, DR Event=No
Period  2: Price=$101.62, DR Prob=0.000, DR Event=No
Period  3: Price=$ 98.38, DR Prob=0.000, DR Event=No
Period  4: Price=$ 97.92, DR Prob=0.000, DR Event=No
Period  5: Price=$ 98.23, DR Prob=0.000, DR Event=No
Period  6: Price=$ 98.88, DR Prob=0.000, DR Event=No
Period  7: Price=$ 95.81, DR Prob=0.000, DR Event=No
Period  8: Price=$ 94.11, DR Prob=0.000, DR Event=No
Period  9: Price=$ 97.87, DR Prob=0.000, DR Event=No
Period 10: Price=$ 98.25, DR Prob=0.000, DR Event=No
... and 38 more periods

Summary:
  Average predicted price: $150.51
  Price range: $94.11 - $610.86
  DR events predicted: 10 out of 48 periods
  DR event rate: 20.8%
